In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

import joblib

In [2]:
df_f = pd.read_parquet('./yellow_tripdata_2023-02.parquet')

# Q1 - Read the data for January. How many columns are there?

In [3]:
#Load Data 
df_j = pd.read_parquet('./yellow_tripdata_2023-01.parquet')
#Print columns
print('There is',df_j.shape[1],'colums in the January file')

There is 19 colums in the January file


# Q2 - What's the standard deviation of the trips duration in January?

In [4]:
df_j['duration'] = df_j.tpep_dropoff_datetime - df_j.tpep_pickup_datetime
df_j['duration'] = df_j.duration.dt.total_seconds() / 60

In [5]:
df_j

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,duration
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.30,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00,8.433333
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.90,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00,6.316667
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.90,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00,12.750000
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.10,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25,9.616667
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.40,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00,10.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3066761,2,2023-01-31 23:58:34,2023-02-01 00:12:33,NaN,3.05,NaN,None,107,48,0,15.80,0.00,0.5,3.96,0.0,1.0,23.76,NaN,NaN,13.983333
3066762,2,2023-01-31 23:31:09,2023-01-31 23:50:36,NaN,5.80,NaN,None,112,75,0,22.43,0.00,0.5,2.64,0.0,1.0,29.07,NaN,NaN,19.450000
3066763,2,2023-01-31 23:01:05,2023-01-31 23:25:36,NaN,4.67,NaN,None,114,239,0,17.61,0.00,0.5,5.32,0.0,1.0,26.93,NaN,NaN,24.516667
3066764,2,2023-01-31 23:40:00,2023-01-31 23:53:00,NaN,3.15,NaN,None,230,79,0,18.15,0.00,0.5,4.43,0.0,1.0,26.58,NaN,NaN,13.000000


In [6]:
mean_jan = df_j['duration'].mean()
print('mean: ', mean_jan)

std_jan = df_j['duration'].std()
print("The standard derivation of the trips duration in January is :", round(std_jan,2), 'minutes')

mean:  15.668995167330449
The standard derivation of the trips duration in January is : 42.59 minutes


# Q3 - What fraction of the records left after you dropped the outliers?

In [7]:
outliers = df_j[(df_j.duration >= 1) & (df_j.duration <= 60)].copy()

In [8]:
frac = outliers.shape[0]/ df_j['duration'].shape[0]*100

print('After the drop of outliers, we kept',frac,'of our database')

After the drop of outliers, we kept 98.1220282212598 of our database


In [9]:
df_j = outliers.copy()

# Q4 - What's the dimensionality of this matrix (number of columns)?

In [10]:
# Create a copy of the Dataframe only with 'tpep_pickup_datetime' and 'tpep_dropoff_datetime'
#df_j = df_j[['tpep_pickup_datetime','tpep_dropoff_datetime']].copy()
X_train = df_j[['PULocationID','DOLocationID']].copy()

In [11]:
# Convert label to str
# df_j['tpep_pickup_datetime'] = df_j['tpep_pickup_datetime'].astype(str)
# df_j['tpep_dropoff_datetime'] = df_j['tpep_dropoff_datetime'].astype(str)
X_train['PULocationID'] = X_train['PULocationID'].astype(str)
X_train['DOLocationID'] = X_train['DOLocationID'].astype(str)

In [12]:
df_j

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,duration
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.30,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00,8.433333
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.90,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00,6.316667
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.90,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00,12.750000
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.10,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25,9.616667
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.40,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00,10.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3066761,2,2023-01-31 23:58:34,2023-02-01 00:12:33,NaN,3.05,NaN,None,107,48,0,15.80,0.00,0.5,3.96,0.0,1.0,23.76,NaN,NaN,13.983333
3066762,2,2023-01-31 23:31:09,2023-01-31 23:50:36,NaN,5.80,NaN,None,112,75,0,22.43,0.00,0.5,2.64,0.0,1.0,29.07,NaN,NaN,19.450000
3066763,2,2023-01-31 23:01:05,2023-01-31 23:25:36,NaN,4.67,NaN,None,114,239,0,17.61,0.00,0.5,5.32,0.0,1.0,26.93,NaN,NaN,24.516667
3066764,2,2023-01-31 23:40:00,2023-01-31 23:53:00,NaN,3.15,NaN,None,230,79,0,18.15,0.00,0.5,4.43,0.0,1.0,26.58,NaN,NaN,13.000000


In [13]:
#Creating the dictionnary
d = X_train.to_dict('records')

In [14]:
v = DictVectorizer()
X_train = v.fit_transform(d)

In [15]:
X_train.shape

(3009173, 515)

# Q5 - Training a model

In [16]:
Y_train = df_j['duration'].values

In [17]:
Y_train

array([ 8.43333333,  6.31666667, 12.75      , ..., 24.51666667,
       13.        , 14.4       ], shape=(3009173,))

In [18]:
reg = LinearRegression().fit(X_train, Y_train)

In [19]:
reg

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [20]:
#saving model
joblib.dump(reg, 'regression_lineaire_100k.joblib')

['regression_lineaire_100k.joblib']

In [17]:
#loading model
reg = joblib.load('regression_lineaire_100k.joblib')

In [22]:
reg.score(X_train, Y_train)

0.4077293835879964

In [27]:
Y_pred = reg.predict(X_train) 

In [24]:
root_mean_squared_error(Y_train, Y_pred)

7.649262444043398

# Q6 - Evaluating the model
## Now let's apply this model to the validation dataset (February 2023).

In [18]:
# Prepare data 
df_f['duration'] = df_f.tpep_dropoff_datetime - df_f.tpep_pickup_datetime
df_f['duration'] = df_f.duration.dt.total_seconds() / 60
print(df_f.shape)
df_f = df_f[(df_f.duration >= 1) & (df_f.duration <= 60)]
print(df_f.shape)
training_df_f = df_f[['PULocationID','DOLocationID']].copy()
print(training_df_f.shape)

(2913955, 20)
(2855951, 20)
(2855951, 2)


In [19]:
training_df_f['PULocationID'] = training_df_f['PULocationID'].astype(str)
training_df_f['DOLocationID'] = training_df_f['DOLocationID'].astype(str)

In [20]:
#Creating the dictionnary
X_train_Feb = training_df_f.to_dict('records')

In [23]:
X_train_Feb

[{'PULocationID': '142', 'DOLocationID': '163'},
 {'PULocationID': '132', 'DOLocationID': '26'},
 {'PULocationID': '161', 'DOLocationID': '145'},
 {'PULocationID': '148', 'DOLocationID': '236'},
 {'PULocationID': '137', 'DOLocationID': '244'},
 {'PULocationID': '263', 'DOLocationID': '141'},
 {'PULocationID': '48', 'DOLocationID': '243'},
 {'PULocationID': '114', 'DOLocationID': '211'},
 {'PULocationID': '114', 'DOLocationID': '249'},
 {'PULocationID': '125', 'DOLocationID': '107'},
 {'PULocationID': '140', 'DOLocationID': '42'},
 {'PULocationID': '140', 'DOLocationID': '226'},
 {'PULocationID': '249', 'DOLocationID': '90'},
 {'PULocationID': '234', 'DOLocationID': '4'},
 {'PULocationID': '114', 'DOLocationID': '125'},
 {'PULocationID': '132', 'DOLocationID': '239'},
 {'PULocationID': '132', 'DOLocationID': '230'},
 {'PULocationID': '140', 'DOLocationID': '68'},
 {'PULocationID': '144', 'DOLocationID': '79'},
 {'PULocationID': '132', 'DOLocationID': '90'},
 {'PULocationID': '236', 'DOL

In [24]:
X_train_Feb = v.transform(X_train_Feb)

In [30]:
Y_train_f = df_f['duration'].values

In [28]:
Y_pred_f = reg.predict(X_train_Feb)

In [31]:
root_mean_squared_error(Y_train_f, Y_pred_f)

7.811812105603362